# Prediction de la pluie en Australie
## Classification Binaire : RainTomorrow

Source : [Kaggle - Weather Dataset Rattle Package](https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package)

**Question :** Etant donne les mesures meteo d'aujourd'hui, est-ce qu'il va pleuvoir demain ?

**Contraintes :** Pas de Deep Learning. Machine Learning classique uniquement.

**Metrique principale :** F2-score (favorise le recall sur la precision).

---

### Comment lire ce notebook ?

Chaque etape contient :
- L'objectif : ce qu'on cherche a faire
- Le concept : pourquoi on fait ca, avec des analogies pour faciliter la comprehension
- Le code : l'implementation
- L'analyse : ce qu'on observe

Note importante : en Machine Learning, 80% du travail est dans la preparation des donnees. Le modele lui-meme n'est que la derniere etape.

## Imports et configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.model_selection import StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, fbeta_score, f1_score, brier_score_loss,
    log_loss, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, make_scorer
)
from sklearn.calibration import calibration_curve

import xgboost as xgb
import lightgbm as lgb

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
RANDOM_STATE = 42
DATA_PATH = '../data/weatherAUS.csv'

# Scorer F2 : le recall compte deux fois plus que la precision
f2_scorer = make_scorer(fbeta_score, beta=2)

print('Imports OK')

---
## Etape 1 - EDA (Analyse Exploratoire des Donnees)

### Objectif

Avant de toucher a un modele, on regarde les donnees. C'est comme lire la recette avant de cuisiner : si on saute cette etape, on risque de se retrouver avec un resultat mediocre sans comprendre pourquoi.

### Concept

L'EDA (Exploratory Data Analysis) repond a ces questions fondamentales :
1. A quoi ressemblent les donnees ? (distributions)
2. Y a-t-il des relations entre les variables ? (correlations)
3. Y a-t-il des patterns dans le temps ? (saisonnalite)
4. Quelle est la qualite des donnees ? (valeurs manquantes, aberrations)

Regle d'or : on ne peut pas bien modeliser ce qu'on ne comprend pas.

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f'Dimensions : {df_raw.shape[0]:,} lignes x {df_raw.shape[1]} colonnes')
df_raw.head(3)

In [ ]:
df_raw.info()

### 1.1 - Valeurs manquantes

La plupart des algorithmes ML ne savent pas gerer les cases vides (NaN). Il faut donc decider pour chaque colonne : supprimer ou remplacer ?

In [ ]:
na_pct = (df_raw.isnull().sum() / len(df_raw) * 100).sort_values(ascending=False)
na_pct = na_pct[na_pct > 0]

fig, ax = plt.subplots(figsize=(13, 4))
colors = ['#DD8452' if v > 30 else '#4C72B0' for v in na_pct.values]
ax.bar(na_pct.index, na_pct.values, color=colors)
ax.axhline(30, color='red', linestyle='--', alpha=0.6, label='Seuil 30%')
ax.set_ylabel('% de valeurs manquantes')
ax.set_title('Pourcentage de valeurs manquantes par colonne')
ax.set_xticklabels(na_pct.index, rotation=45, ha='right')
ax.legend()
for i, (col, val) in enumerate(na_pct.items()):
    ax.text(i, val + 0.5, f'{val:.0f}%', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

print(na_pct)

**Analyse :**

- `Sunshine` (48%) et `Evaporation` (43%) ont plus de 40% de NaN. Ce sont pourtant des variables tres informatives pour la pluie. On les conserve et on imputera par la mediane.
- Les autres colonnes (moins de 11% de NaN) : imputation simple suffisante.
- Strategie adoptee : on ne supprime aucune colonne a cause des NaN, on impute intelligemment.

### 1.2 - Distribution de la cible : le desequilibre de classes

Imaginez un medecin qui diagnostique une maladie rare touchant 1% de la population. Un modele qui dit "tout le monde est en bonne sante" aurait 99% d'accuracy... mais serait completement inutile. C'est exactement le meme probleme ici.

C'est pourquoi l'accuracy est une mauvaise metrique pour ce probleme. On utilisera le F2-score et le Brier Score a la place.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df_raw['RainTomorrow'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#4C72B0', '#DD8452'], alpha=0.85)
axes[0].set_title('Distribution de RainTomorrow')
axes[0].set_xlabel('Pluie demain ?')
axes[0].set_ylabel('Nombre de jours')
for i, (k, v) in enumerate(counts.items()):
    axes[0].text(i, v + 600, f'{v:,}\n({v/len(df_raw)*100:.1f}%)',
                 ha='center', fontweight='bold', fontsize=11)

axes[1].pie(counts.values, labels=['No Rain', 'Rain'],
            autopct='%1.1f%%', colors=['#4C72B0', '#DD8452'],
            startangle=90, explode=(0, 0.1))
axes[1].set_title(f'Ratio No/Yes = {counts["No"]/counts["Yes"]:.1f} : 1')

plt.suptitle('Desequilibre de classes : 3.5x plus de No Rain que Rain', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Le modele naif ("il ne pleut jamais") aurait {counts["No"]/counts.sum()*100:.1f}% d\'accuracy.')
print('L\'accuracy est donc une mauvaise metrique pour ce probleme.')

### 1.3 - Saisonnalite

La meteo n'est pas aleatoire dans le temps. En Australie, l'hiver austral (juin-aout) correspond a la saison des pluies dans le sud. Si on ignore ce pattern temporel, le modele sera incapable de le capturer.

In [ ]:
df_eda = df_raw.copy()
df_eda['Date']  = pd.to_datetime(df_eda['Date'])
df_eda['Month'] = df_eda['Date'].dt.month
df_eda['Year']  = df_eda['Date'].dt.year
df_eda['RainTomorrow_num'] = (df_eda['RainTomorrow'] == 'Yes').astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

monthly = df_eda.groupby('Month')['RainTomorrow_num'].mean() * 100
axes[0].plot(monthly.index, monthly.values, marker='o', color='#4C72B0', linewidth=2.5)
axes[0].fill_between(monthly.index, monthly.values, alpha=0.15, color='#4C72B0')
axes[0].set_title('Taux de pluie mensuel')
axes[0].set_xlabel('Mois')
axes[0].set_ylabel('% RainTomorrow = Yes')
axes[0].set_xticks(range(1,13))
axes[0].set_xticklabels(['Jan','Fev','Mar','Avr','Mai','Jun','Jul','Aou','Sep','Oct','Nov','Dec'])
axes[0].grid(alpha=0.4)

location_rain = df_eda.groupby('Location')['RainTomorrow_num'].mean().sort_values(ascending=False).head(12) * 100
axes[1].barh(location_rain.index[::-1], location_rain.values[::-1], color='#DD8452', alpha=0.85)
axes[1].set_title('Top 12 localites les plus pluvieuses')
axes[1].set_xlabel('% RainTomorrow = Yes')
axes[1].axvline(monthly.mean(), color='navy', linestyle='--', alpha=0.5,
                label=f'Moyenne : {monthly.mean():.1f}%')
axes[1].legend()

plt.tight_layout()
plt.show()

### 1.4 - Correlations avec la cible

In [ ]:
df_corr = df_eda.copy()
df_corr['RainToday_num'] = (df_corr['RainToday'] == 'Yes').astype(int)

num_cols = df_corr.select_dtypes(include='float64').columns.tolist() + ['RainToday_num', 'RainTomorrow_num']
num_cols = list(dict.fromkeys(num_cols))

corr_matrix = df_corr[num_cols].corr()
corr_target = corr_matrix['RainTomorrow_num'].drop('RainTomorrow_num')
corr_target = pd.Series(corr_target.values, index=corr_target.index).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

colors = ['#DD8452' if v > 0 else '#4C72B0' for v in corr_target.values]
corr_target.plot(kind='barh', ax=axes[0], color=colors, alpha=0.85)
axes[0].axvline(0, color='black', lw=0.8)
axes[0].set_title('Correlation avec RainTomorrow')
axes[0].set_xlabel('Coefficient de Pearson')

key_cols = ['Humidity3pm','Humidity9am','Cloud3pm','Cloud9am','Sunshine',
            'Pressure9am','Pressure3pm','Rainfall','RainToday_num','RainTomorrow_num']
key_cols = [c for c in key_cols if c in df_corr.columns]
sns.heatmap(df_corr[key_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=axes[1], square=True, linewidths=0.5)
axes[1].set_title('Heatmap des correlations (variables cles)')

plt.tight_layout()
plt.show()

print('Variables les plus liees positivement a la pluie demain :')
print(corr_target.tail(4).to_string())
print('\nVariables les plus liees negativement a la pluie demain :')
print(corr_target.head(4).to_string())

### 1.5 - Distributions par classe

In [ ]:
key_num = ['Humidity3pm', 'Pressure9am', 'Cloud3pm', 'Sunshine', 'Rainfall', 'WindGustSpeed']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, col in zip(axes.flatten(), key_num):
    if col not in df_eda.columns:
        continue
    for label, color, name in [(0,'#4C72B0','No Rain'), (1,'#DD8452','Rain')]:
        data = df_eda[df_eda['RainTomorrow_num'] == label][col].dropna()
        ax.hist(data, bins=40, alpha=0.55, color=color, label=name, density=True)
    ax.set_title(col, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Distribution des variables cles selon RainTomorrow', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Observations :')
print('- Humidity3pm > 70% : tres souvent associe a la pluie')
print('- Pressure9am < 1010 hPa : zone de basse pression = front pluvieux')
print('- Sunshine < 4h : journee couverte = risque de pluie')
print('- Cloud3pm > 6 : ciel tres couvert = pluie imminente')

---
## Etape 2 - Cleaning

### Objectif

Nettoyer les donnees sans en supprimer trop. Chaque ligne supprimee est une information perdue.

### Concept : que supprimer ?

Il y a deux types de "mauvaises" donnees :

**Les aberrations physiques** : valeurs impossibles dans la realite (humidite a 120%, pression a -50 hPa). Ce sont des erreurs de capteur a supprimer.

**Les outliers statistiques** : valeurs extremes mais physiquement possibles (vent a 130 km/h = cyclone rare mais reel). On les garde. Le RobustScaler les gerera.

Exemple : si un releve dit qu'il faisait 200 degres a Melbourne, c'est une erreur de thermometre. Mais si ca dit 48 degres, c'est chaud mais possible (record Australie : 50.7 degres).

In [ ]:
df = pd.read_csv(DATA_PATH)
n_initial = len(df)

# Doublons
n_dup = df.duplicated().sum()
df = df.drop_duplicates()

# Date -> datetime
df['Date'] = pd.to_datetime(df['Date'])

# Aberrations physiques uniquement
aberrations = {
    'Humidity9am <= 100%':    lambda d: (d['Humidity9am'].isna()) | (d['Humidity9am'] <= 100),
    'Humidity3pm <= 100%':    lambda d: (d['Humidity3pm'].isna()) | (d['Humidity3pm'] <= 100),
    'Rainfall >= 0 mm':       lambda d: (d['Rainfall'].isna())    | (d['Rainfall'] >= 0),
    'Pressure9am > 800 hPa':  lambda d: (d['Pressure9am'].isna()) | (d['Pressure9am'] > 800),
    'Pressure3pm > 800 hPa':  lambda d: (d['Pressure3pm'].isna()) | (d['Pressure3pm'] > 800),
    'WindSpeed9am >= 0':      lambda d: (d['WindSpeed9am'].isna())| (d['WindSpeed9am'] >= 0),
    'WindSpeed3pm >= 0':      lambda d: (d['WindSpeed3pm'].isna())| (d['WindSpeed3pm'] >= 0),
}
for rule_name, rule_fn in aberrations.items():
    n_before = len(df)
    df = df[rule_fn(df)]
    removed = n_before - len(df)
    if removed > 0:
        print(f'  Regle [{rule_name}] : {removed} lignes supprimees')

# Lignes sans target (irrécuperables)
n_before = len(df)
df = df.dropna(subset=['RainTomorrow'])
print(f'  Lignes sans target : {n_before - len(df)} supprimees')

# Encoder la target
df['RainTomorrow'] = (df['RainTomorrow'] == 'Yes').astype(int)
df['RainToday']    = df['RainToday'].map({'Yes': 1, 'No': 0})

print(f'\nDoublons supprimes : {n_dup}')
print(f'Total supprime : {n_initial - len(df)} lignes ({(n_initial-len(df))/n_initial*100:.1f}%)')
print(f'Shape finale : {df.shape}')
print(f'Target : {df["RainTomorrow"].value_counts().to_dict()}')

---
## Etape 3 - Feature Engineering

### Objectif

Creer de nouvelles variables a partir des variables existantes pour donner plus d'information au modele.

### Pourquoi creer des features ?

Les modeles ML ne comprennent pas la physique. Ils ne savent pas que :
- Une chute de pression est signe de pluie imminente
- Une humidite croissante toute la journee indique une saturation atmospherique
- Le mois de juillet en Australie correspond a l'hiver austral et donc plus de pluie dans le sud

En creant ces variables explicitement, on encode la connaissance meteorologique dans les donnees.

### L'encodage cyclique (sin/cos)

Le mois de janvier (1) et decembre (12) sont proches dans le calendrier mais loin numeriquement. Un modele qui recoit Month=12 et Month=1 ne sait pas qu'ils sont adjacents. La solution est d'encoder le mois en cercle :

    Month_sin = sin(2 * pi * Month / 12)
    Month_cos = cos(2 * pi * Month / 12)

Avec cet encodage, janvier et decembre sont au meme endroit sur le cercle. Le modele comprend leur proximite.

In [ ]:
# Illustration de l'encodage cyclique
months = np.arange(1, 13)
sin_vals = np.sin(2 * np.pi * months / 12)
cos_vals = np.cos(2 * np.pi * months / 12)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(months, sin_vals, 'o-', color='#4C72B0', label='sin')
axes[0].plot(months, cos_vals, 's-', color='#DD8452', label='cos')
axes[0].set_xticks(months)
axes[0].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
axes[0].set_title('Encodage sin/cos du mois')
axes[0].legend()
axes[0].grid(alpha=0.4)

theta = 2 * np.pi * months / 12
axes[1].scatter(np.cos(theta), np.sin(theta), s=100, c=months, cmap='hsv', zorder=3)
for m, t in zip(months, theta):
    axes[1].text(np.cos(t)*1.15, np.sin(t)*1.15,
                 ['J','F','M','A','M','J','J','A','S','O','N','D'][m-1], ha='center', fontsize=9)
circle = plt.Circle((0, 0), 1, fill=False, linestyle='--', alpha=0.5)
axes[1].add_patch(circle)
axes[1].set_xlim(-1.4, 1.4)
axes[1].set_ylim(-1.4, 1.4)
axes[1].set_aspect('equal')
axes[1].set_title('Les mois vus comme un cercle (Jan et Dec sont voisins)')
axes[1].grid(alpha=0.4)

plt.tight_layout()
plt.show()
print('Janvier et decembre sont au meme endroit sur le cercle.')
print('Le modele comprend donc qu\'ils sont voisins temporellement.')

In [ ]:
# Trier par Location + Date (obligatoire pour les calculs temporels)
df = df.sort_values(['Location', 'Date']).reset_index(drop=True)

# Variables temporelles
df['Month']     = df['Date'].dt.month
df['DayOfYear'] = df['Date'].dt.dayofyear
df['Season']    = df['Month'].map({
    12:'Summer',1:'Summer',2:'Summer',3:'Autumn',4:'Autumn',5:'Autumn',
    6:'Winter',7:'Winter',8:'Winter',9:'Spring',10:'Spring',11:'Spring'
})
df['Month_sin']     = np.sin(2*np.pi*df['Month']/12)
df['Month_cos']     = np.cos(2*np.pi*df['Month']/12)
df['DayOfYear_sin'] = np.sin(2*np.pi*df['DayOfYear']/365)
df['DayOfYear_cos'] = np.cos(2*np.pi*df['DayOfYear']/365)

# Derivees temporelles 9h -> 15h (evolution dans la journee)
# Un delta negatif de pression signifie que la pression baisse = front meteorologique approchant
df['Delta_Pressure'] = df['Pressure3pm'] - df['Pressure9am']
df['Delta_Humidity'] = df['Humidity3pm'] - df['Humidity9am']
df['Delta_Temp']     = df['Temp3pm']     - df['Temp9am']
df['Delta_Wind']     = df['WindSpeed3pm']- df['WindSpeed9am']

# Variation par rapport au jour precedent (J-1)
for col in ['Pressure9am', 'Humidity3pm', 'MaxTemp']:
    lag = df.groupby('Location')[col].shift(1)
    df[f'{col}_diff1'] = df[col] - lag

# Fenetres glissantes 3 jours - on utilise shift(1) pour ne regarder que le passe
# Important : sans shift(1), on incorporerait la valeur du jour J dans la moyenne
# ce qui creeraitune fuite de donnees
for col in ['Rainfall', 'Humidity3pm', 'Pressure9am']:
    df[f'{col}_roll3_mean'] = df.groupby('Location')[col].transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).mean()
    )
    df[f'{col}_roll3_max'] = df.groupby('Location')[col].transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).max()
    )

# Flags binaires : signaux meteorologiques discrets
df['Pressure_drop_flag']  = (df['Delta_Pressure'] < -2).astype(int)
df['HighHumidity_flag']   = (df['Humidity3pm'] > 85).astype(int)
df['StrongWind_flag']     = (df['WindGustSpeed'] > 60).astype(int)
df['HumidityRising_flag'] = (df['Delta_Humidity'] > 10).astype(int)
df['PressureLow_flag']    = (df['Pressure9am'] < 1010).astype(int)

# Interactions
df['Rain_x_Humidity'] = df['RainToday'].fillna(0) * df['Humidity3pm'].fillna(df['Humidity3pm'].median())
df['Wind_x_Humidity'] = df['WindGustSpeed'].fillna(0) * df['Humidity3pm'].fillna(df['Humidity3pm'].median())
df['TempRange']       = df['MaxTemp'] - df['MinTemp']

# Encodage cyclique des directions de vent
# Probleme identique au mois : le Nord (0 degres) et le NNW (337.5 degres) sont voisins
wind_dir_map = {
    'N':0,'NNE':22.5,'NE':45,'ENE':67.5,'E':90,'ESE':112.5,'SE':135,'SSE':157.5,
    'S':180,'SSW':202.5,'SW':225,'WSW':247.5,'W':270,'WNW':292.5,'NW':315,'NNW':337.5
}
for col in ['WindGustDir', 'WindDir9am', 'WindDir3pm']:
    angles = df[col].map(wind_dir_map)
    df[f'{col}_sin'] = np.sin(np.radians(angles))
    df[f'{col}_cos'] = np.cos(np.radians(angles))

df = df.drop(columns=['WindGustDir', 'WindDir9am', 'WindDir3pm'])

new_features = [c for c in df.columns if any(x in c for x in
    ['Delta','flag','roll3','sin','cos','diff1','Range','x_'])]
print(f'{len(new_features)} nouvelles features creees :')
for f in sorted(new_features):
    print(f'  - {f}')
print(f'\nShape totale : {df.shape}')

---
## Etape 4 - Split Train / Test CHRONOLOGIQUE

### Objectif

Separer les donnees en deux groupes sans regarder le futur depuis le passe.

### Pourquoi un split chronologique et pas aleatoire ?

C'est l'une des decisions les plus importantes du projet. Voici pourquoi.

Un split aleatoire (train_test_split avec random_state) sur des donnees temporelles est une erreur fondamentale. Le modele peut s'entrainer sur des donnees de 2016 et etre evalue sur des donnees de 2009. Il a donc "vu le futur" pendant l'entrainement. Les metriques sont alors artificiellement gonflees - elles ne refletent pas la vraie capacite du modele a predire des jours qu'il n'a jamais vus.

Illustration :

    Split aleatoire (incorrect pour donnees temporelles)
    2007  2010  2013  2008  2015  2011  2016  2009
     [T]   [Ts]  [T]  [Ts]   [T]  [Ts]   [T]  [Ts]
    (T=train, Ts=test, melanges aleatoirement)

    Split chronologique (correct)
    2007  2008  2009  2010  2011  2012  2013  2014 | 2015  2016  2017
    <------------- TRAIN (80%) -------------------> <-- TEST (20%) -->

Les metriques avec split chronologique sont generalement plus basses qu'avec split aleatoire. Mais elles sont honnetes, ce qui est ce qui compte.

In [ ]:
# Coupure sur les 80% de dates uniques
unique_dates = df['Date'].sort_values().unique()
cutoff_date  = unique_dates[int(len(unique_dates) * 0.8)]

train_mask = df['Date'] <  cutoff_date
test_mask  = df['Date'] >= cutoff_date

print(f'Date de coupure : {pd.Timestamp(cutoff_date).date()}')
print(f'Train : {df["Date"].min().date()} -> {pd.Timestamp(cutoff_date).date()} ({train_mask.sum():,} obs.)')
print(f'Test  : {pd.Timestamp(cutoff_date).date()} -> {df["Date"].max().date()} ({test_mask.sum():,} obs.)')

# Dropper la Date de X (les features cycliques la representent deja)
df = df.drop(columns=['Date'])

X = df.drop(columns=['RainTomorrow'])
y = df['RainTomorrow']

X_train = X[train_mask].reset_index(drop=True)
X_test  = X[test_mask].reset_index(drop=True)
y_train = y[train_mask].reset_index(drop=True)
y_test  = y[test_mask].reset_index(drop=True)

print(f'\nTrain : {X_train.shape} | ratio pluie : {y_train.mean():.3f} ({y_train.mean()*100:.1f}%)')
print(f'Test  : {X_test.shape}  | ratio pluie : {y_test.mean():.3f} ({y_test.mean()*100:.1f}%)')
print(f'\nA partir d\'ici, X_test ne sera utilise qu\'a l\'etape 12 pour l\'evaluation finale.')

fig, ax = plt.subplots(figsize=(12, 2.5))
ax.barh([''], [train_mask.sum()], color='#4C72B0', alpha=0.8, label=f'Train ({train_mask.sum():,})')
ax.barh([''], [test_mask.sum()], left=[train_mask.sum()], color='#DD8452', alpha=0.8,
        label=f'Test ({test_mask.sum():,})')
ax.set_title('Split chronologique Train / Test')
ax.legend(loc='center')
ax.set_xlabel('Nombre de jours')
plt.tight_layout()
plt.show()

---
## Etape 5 - Preprocessing (Pipeline sklearn)

### Objectif

Preparer les donnees numeriquement pour que les algorithmes puissent les traiter correctement.

### Pourquoi normaliser ?

La temperature varie entre -8 et +48 degres Celsius. La pression entre 980 et 1041 hPa. Si on ne normalise pas, la pression ecrase numeriquement la temperature dans les calculs mathematiques, non pas parce qu'elle est plus importante, mais simplement parce qu'elle a des valeurs plus grandes.

### Pourquoi RobustScaler plutot que StandardScaler ?

    StandardScaler : (x - moyenne) / ecart-type
    Probleme : la moyenne et l'ecart-type sont tres sensibles aux valeurs extremes.

    RobustScaler  : (x - mediane) / (Q75 - Q25)
    Avantage : utilise la mediane et l'IQR, qui sont resistants aux outliers.

En meteorologie, les cyclones, canicules et tempetes creent des valeurs extremes reelles. RobustScaler est donc plus adapte.

### Pourquoi un Pipeline ?

Le Pipeline sklearn garantit que toute transformation (normalisation, imputation) est apprise sur le train uniquement, puis appliquee au test. Sans pipeline, on risque d'apprendre la normalisation sur l'ensemble du dataset, ce qui constitue une fuite de donnees (data leakage).

In [ ]:
numeric_features     = X_train.select_dtypes(include=['float64','int64','int32']).columns.tolist()
categorical_features = X_train.select_dtypes(include='object').columns.tolist()

print(f'Variables numeriques ({len(numeric_features)}) : {numeric_features}')
print(f'\nVariables categorielles ({len(categorical_features)}) : {categorical_features}')

preprocessor = ColumnTransformer(transformers=[
    # Pipeline variables numeriques :
    # 1) Imputer par la mediane (robuste aux outliers) pour les NaN
    # 2) RobustScaler pour normaliser
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler())
    ]), numeric_features),

    # Pipeline variables categorielles :
    # 1) Imputer par le mode pour les NaN
    # 2) OneHotEncoder : cree une colonne binaire par categorie
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), categorical_features)
], remainder='drop')

print('\nPreprocessor defini. Le pipeline garantit l\'absence de data leakage.')

---
## Etape 6 - Ponderation anti-label-noise

### Objectif

Reduire l'influence des observations ambigues ou mal etiquetees sur l'apprentissage.

### Le bruit de label en meteorologie

Les donnees meteo sont imparfaites. Deux situations typiques :

**Cas 1 :** 0.2mm de pluie est tombe hier. RainToday = 'Yes' (compte-goutte sur le pluviometre). Mais est-ce vraiment "de la pluie" au sens meteorologique ? Ce jour est ambigu. Garder son poids fort va perturber le modele.

**Cas 2 :** 5mm de pluie aujourd'hui + RainTomorrow = 'Yes'. C'est un cas clairement positif. On lui donne plus de poids pour que le modele apprenne correctement de lui.

C'est comme reviser pour un examen : on insiste sur les exercices clairs et on ne s'obstine pas sur les exercices avec des enondes incomprehensibles.

| Cas | Poids | Raison |
|-----|-------|--------|
| Rainfall < 0.5mm et Rain=1 | 0.6 | Pluie trop faible pour etre fiable |
| Humidity entre 60 et 75% et Rain=0 | 0.8 | Zone ambigue de l'humidite |
| Rainfall > 5mm et Rain=1 | 1.5 | Cas clairement positif |
| Autres | 1.0 | Neutre |

In [ ]:
rainfall_train = X_train['Rainfall'].fillna(0)
humidity_train = X_train['Humidity3pm'].fillna(50)

sample_weight = np.ones(len(y_train))

ambig_low = (rainfall_train < 0.5) & (y_train == 1)
ambig_mid = humidity_train.between(60, 75) & (y_train == 0)
clear_pos = (rainfall_train > 5) & (y_train == 1)

sample_weight[ambig_low.values] = 0.6
sample_weight[ambig_mid.values] = 0.8
sample_weight[clear_pos.values] = 1.5

print('Distribution des poids :')
print(f'  Poids 0.6 (pluie faible, label ambigu)    : {ambig_low.sum():,} observations')
print(f'  Poids 0.8 (humidite moyenne, zone grise)  : {ambig_mid.sum():,} observations')
print(f'  Poids 1.0 (neutre)                        : {(sample_weight == 1.0).sum():,} observations')
print(f'  Poids 1.5 (pluie forte, cas clair)        : {clear_pos.sum():,} observations')
print(f'  Poids moyen : {sample_weight.mean():.3f}')

fig, ax = plt.subplots(figsize=(8, 3))
unique_w, counts_w = np.unique(sample_weight, return_counts=True)
colors_w = ['#C44E52','#DD8452','#4C72B0','#55A868']
bars = ax.bar([str(w) for w in unique_w], counts_w, color=colors_w, alpha=0.85)
for bar, c in zip(bars, counts_w):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100, f'{c:,}', ha='center', fontsize=9)
ax.set_title('Distribution des sample_weights')
ax.set_xlabel('Poids attribue')
ax.set_ylabel('Nb observations')
plt.tight_layout()
plt.show()

---
## Etape 7 - Feature Selection automatique

### Objectif

Garder uniquement les variables les plus informatives et reduire le bruit.

### Mutual Information vs Correlation de Pearson

La correlation de Pearson ne mesure que les relations lineaires. Si l'humidite a 90% provoque la pluie de maniere non-lineaire (effet seuil), la correlation ne le verra pas.

L'Information Mutuelle (Mutual Information) mesure la dependance generale entre deux variables, quelle que soit sa forme. C'est la quantite d'information qu'une variable donne sur une autre.

Pour faire simple : la correlation de Pearson demande "sont-ils proportionnels ?". L'Information Mutuelle demande "est-ce qu'en connaissant X, j'en sais plus sur Y ?"

In [ ]:
# Calcul des scores de Mutual Information pour visualisation
prep_temp = ColumnTransformer(transformers=[
    ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), numeric_features),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]),
     categorical_features)
])

X_prep_temp = prep_temp.fit_transform(X_train)
ohe_cols    = prep_temp.named_transformers_['cat']['ohe'].get_feature_names_out(categorical_features)
all_feat    = np.array(numeric_features + list(ohe_cols))

mi_scores = mutual_info_classif(X_prep_temp, y_train, random_state=RANDOM_STATE)
mi_df = pd.DataFrame({'feature': all_feat, 'mi_score': mi_scores})
mi_df = mi_df.sort_values('mi_score', ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
top20 = mi_df.head(20)
colors_mi = ['#DD8452' if i < 5 else '#4C72B0' for i in range(20)]
ax.barh(top20['feature'][::-1], top20['mi_score'][::-1], color=colors_mi[::-1], alpha=0.85)
ax.set_title('Top 20 features par Information Mutuelle avec RainTomorrow\n(en orange : top 5 les plus informatives)', fontsize=11)
ax.set_xlabel('Score d\'Information Mutuelle')
plt.tight_layout()
plt.show()

print('Top 10 features les plus informatives :')
for _, row in mi_df.head(10).iterrows():
    print(f'  {row["feature"]:<30} MI = {row["mi_score"]:.4f}')

---
## Etape 8 - Pas de SMOTE

### Pourquoi ne pas utiliser SMOTE en meteorologie ?

SMOTE (Synthetic Minority Over-sampling Technique) cree des observations synthetiques en interpolant entre des observations existantes de la classe minoritaire.

En meteorologie, cela pose deux problemes fondamentaux :

**Probleme 1 - Non physique :** SMOTE cree une observation "entre" deux jours de pluie reels. Cette observation n'existe pas dans la realite. Ce n'est pas un etat physique de l'atmosphere.

    Observation reelle A : Pression=1005, Humidite=85%, Vent=45 km/h -> Pluie
    Observation reelle B : Pression=1012, Humidite=72%, Vent=20 km/h -> Pluie
    SMOTE genere :         Pression=1008, Humidite=78%, Vent=32 km/h -> Pluie ???

**Probleme 2 - Structure temporelle :** Les donnees meteo sont correlees dans le temps. SMOTE cree un faux "jour" qui brise cette structure temporelle.

**Notre alternative :** class_weight='balanced' dans les modeles, et scale_pos_weight pour XGBoost, combines avec nos sample_weights definis en etape 6.

Regle generale : pour des donnees temporelles ou des donnees physiques (meteo, hydrologie, finance), SMOTE est deconseille.

---
## Etape 9 - Training : Comparaison de modeles

### Objectif

Evaluer plusieurs modeles en cross-validation pour identifier le plus prometteur.

### La Cross-Validation Stratifiee (5-fold)

Au lieu d'entrainer une seule fois et tester une seule fois (ce qui peut etre du au hasard du split), la CV k-fold :

1. Divise le train en 5 groupes (folds)
2. Entraine sur 4 folds, teste sur le 5eme
3. Repete 5 fois en changeant le fold de test
4. Retourne la moyenne et l'ecart-type des 5 scores

"Stratifiee" : chaque fold conserve le meme ratio No Rain / Rain que le dataset complet, ce qui est essentiel avec des classes desequilibrees.

### Pourquoi F2-score plutot que F1-score ?

Le F1-score donne le meme poids a la precision et au recall :

    F1 = 2 * (Precision * Recall) / (Precision + Recall)

Mais les erreurs ne coutent pas pareil en meteorologie :

- **Faux Negatif** (on predit "pas de pluie", il pleut) : l'agriculteur ne protege pas sa recolte, les travaux en exterieur ne sont pas planifies en consequence. Cout eleve.
- **Faux Positif** (on predit "pluie", il fait beau) : on prend un parapluie inutilement. Cout faible.

Le F2-score penalise davantage les Faux Negatifs. Le recall compte deux fois plus que la precision :

    F2 = 5 * (Precision * Recall) / (4 * Precision + Recall)

Regle generale : utilisez F-beta avec beta > 1 quand manquer un positif est plus grave que declencher une fausse alarme.

In [ ]:
# Illustration : F1 vs F2 avec differents profils de modeles
cases = [
    ('Modele A (equilibre)',        0.70, 0.70),
    ('Modele B (precision forte)',  0.85, 0.55),
    ('Modele C (recall fort)',      0.55, 0.85),
]
print('Comparaison F1 vs F2 sur 3 profils de modeles :')
print(f'{"Modele":<30} {"Precision":>10} {"Recall":>8} {"F1":>8} {"F2":>8}')
print('-' * 65)
for name, p, r in cases:
    f1 = 2*p*r/(p+r)
    f2 = 5*p*r/(4*p+r)
    note = '  <- F2 prefere ce modele' if name == 'Modele C (recall fort)' else ''
    print(f'{name:<30} {p:>10.0%} {r:>8.0%} {f1:>8.4f} {f2:>8.4f}{note}')

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    'RandomForest': Pipeline([
        ('preprocessor', preprocessor),
        ('selector', SelectKBest(score_func=mutual_info_classif, k=35)),
        ('model', RandomForestClassifier(
            n_estimators=300, max_depth=15, min_samples_split=5,
            min_samples_leaf=2, class_weight='balanced',
            random_state=RANDOM_STATE, n_jobs=-1
        ))
    ]),
    'XGBoost': Pipeline([
        ('preprocessor', preprocessor),
        ('selector', SelectKBest(score_func=mutual_info_classif, k=35)),
        ('model', xgb.XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
            eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
        ))
    ]),
    'LightGBM': Pipeline([
        ('preprocessor', preprocessor),
        ('selector', SelectKBest(score_func=mutual_info_classif, k=35)),
        ('model', lgb.LGBMClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
        ))
    ])
}

cv_results = {}
for name, pipeline in models.items():
    print(f'CV {name}...', end=' ', flush=True)
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring=f2_scorer, n_jobs=-1)
    cv_results[name] = scores
    print(f'F2 = {scores.mean():.4f} +/- {scores.std():.4f}')

best_cv = max(cv_results, key=lambda k: cv_results[k].mean())
print(f'\nMeilleur modele en CV : {best_cv}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

names   = list(cv_results.keys())
means   = [cv_results[n].mean() for n in names]
stds    = [cv_results[n].std() for n in names]
cols_bar = ['#DD8452' if m == max(means) else '#4C72B0' for m in means]

bars = axes[0].bar(names, means, yerr=stds, capsize=6, color=cols_bar, alpha=0.85)
axes[0].set_title('Comparaison des modeles (F2-score, CV 5-fold)')
axes[0].set_ylabel('F2-score')
axes[0].set_ylim(min(means) - 0.05, max(means) + 0.05)
for bar, m, s in zip(bars, means, stds):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.002,
                 f'{m:.4f}', ha='center', fontweight='bold', fontsize=10)

data_box = [cv_results[n] for n in names]
bp = axes[1].boxplot(data_box, labels=names, patch_artist=True)
for patch, c in zip(bp['boxes'], cols_bar):
    patch.set_facecolor(c)
    patch.set_alpha(0.7)
axes[1].set_title('Distribution des scores CV (stabilite)')
axes[1].set_ylabel('F2-score par fold')
axes[1].grid(alpha=0.4)

plt.suptitle('Comparaison des modeles - Split chronologique', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Etape 10 - Optimisation des Hyperparametres (RandomizedSearchCV)

### Objectif

Trouver les meilleurs reglages pour les deux modeles finalistes.

### GridSearch vs RandomizedSearch

GridSearchCV essaie toutes les combinaisons possibles. Si on a 5 valeurs pour 8 parametres differents, c'est 5^8 = 390 625 combinaisons. C'est inutilement long.

RandomizedSearchCV tire aleatoirement N combinaisons. 30 a 50 iterations suffisent generalement pour trouver une bonne region de l'espace des hyperparametres. C'est le meilleur compromis temps/qualite.

Point important : le nombre de features k de la SelectionKBest est inclus dans l'espace de recherche. On optimise le pipeline complet, pas seulement le modele.

In [ ]:
# Tuning LightGBM
lgbm_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('selector', SelectKBest(score_func=mutual_info_classif, k=35)),
    ('model', lgb.LGBMClassifier(
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    ))
])
lgbm_params = {
    'selector__k':              [25, 30, 35, 40],
    'model__n_estimators':      [200, 300, 400, 500],
    'model__max_depth':         [4, 5, 6, 7, 8],
    'model__learning_rate':     [0.03, 0.05, 0.07, 0.1],
    'model__num_leaves':        [31, 50, 63, 80],
    'model__subsample':         [0.7, 0.8, 0.9],
    'model__colsample_bytree':  [0.7, 0.8, 0.9],
    'model__reg_alpha':         [0, 0.1, 0.5],
    'model__reg_lambda':        [0, 0.1, 0.5, 1.0],
    'model__min_child_samples': [10, 20, 30],
}
print('RandomizedSearch LightGBM (30 iterations, scoring=F2)...')
lgbm_search = RandomizedSearchCV(
    lgbm_pipe, lgbm_params, n_iter=30, cv=cv,
    scoring=f2_scorer, n_jobs=-1, random_state=RANDOM_STATE, verbose=0
)
lgbm_search.fit(X_train, y_train)
print(f'LightGBM best F2 (CV) : {lgbm_search.best_score_:.4f}')
print('Meilleurs parametres :')
for k, v in lgbm_search.best_params_.items():
    print(f'  {k}: {v}')

In [ ]:
# Tuning XGBoost
xgb_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('selector', SelectKBest(score_func=mutual_info_classif, k=35)),
    ('model', xgb.XGBClassifier(
        scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
        eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
    ))
])
xgb_params = {
    'selector__k':             [25, 30, 35, 40],
    'model__n_estimators':     [200, 300, 400, 500],
    'model__max_depth':        [4, 5, 6, 7],
    'model__learning_rate':    [0.03, 0.05, 0.07, 0.1],
    'model__subsample':        [0.7, 0.8, 0.9],
    'model__colsample_bytree': [0.7, 0.8, 0.9],
    'model__gamma':            [0, 0.1, 0.5],
    'model__reg_alpha':        [0, 0.1, 0.5],
    'model__reg_lambda':       [0.5, 1.0, 1.5],
    'model__min_child_weight': [1, 3, 5],
}
print('RandomizedSearch XGBoost (30 iterations, scoring=F2)...')
xgb_search = RandomizedSearchCV(
    xgb_pipe, xgb_params, n_iter=30, cv=cv,
    scoring=f2_scorer, n_jobs=-1, random_state=RANDOM_STATE, verbose=0
)
xgb_search.fit(X_train, y_train)
print(f'XGBoost best F2 (CV) : {xgb_search.best_score_:.4f}')
print('Meilleurs parametres :')
for k, v in xgb_search.best_params_.items():
    print(f'  {k}: {v}')

In [ ]:
# Selection du meilleur modele
if lgbm_search.best_score_ >= xgb_search.best_score_:
    best_search, best_name = lgbm_search, 'LightGBM'
else:
    best_search, best_name = xgb_search, 'XGBoost'

print(f'Meilleur modele tunne : {best_name} (F2 CV = {best_search.best_score_:.4f})')

# Visualisation de la recherche
cv_res_df = pd.DataFrame(best_search.cv_results_).sort_values('rank_test_score')

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(range(len(cv_res_df)), cv_res_df['mean_test_score'],
           alpha=0.5, color='#4C72B0', s=60)
ax.axhline(best_search.best_score_, color='red', linestyle='--', lw=2,
           label=f'Meilleur F2 = {best_search.best_score_:.4f}')
ax.set_xlabel('Rang (meilleur -> moins bon)')
ax.set_ylabel('F2-score CV')
ax.set_title(f'Resultats RandomizedSearchCV - {best_name}')
ax.legend()
ax.grid(alpha=0.4)
plt.tight_layout()
plt.show()

---
## Etape 11 - Entrainement Final

### Objectif

Reentrainer le meilleur modele tunne sur l'integralite du train.

### Pourquoi reentrainer ?

Pendant le RandomizedSearchCV, chaque combinaison de parametres etait evaluee sur des sous-ensembles du train (les folds). Le modele final utilise maintenant 100% des donnees d'entrainement, ce qui lui permet d'apprendre davantage de patterns.

In [ ]:
final_model = best_search.best_estimator_
final_model.fit(X_train, y_train)
print(f'Modele final ({best_name}) reentrainee sur {len(X_train):,} observations.')
print(f'Parametres : {best_search.best_params_}')

---
## Etape 12 - Evaluation Finale

### Objectif

Mesurer les vraies performances sur le test set, jamais vu pendant l'entrainement.

### Les metriques utilisees

#### F2-score (metrique principale)
Comme explique en etape 9 : le recall compte deux fois plus que la precision.

#### Brier Score

C'est la metrique des meteorologues professionnels. Elle mesure la qualite des probabilites predites, pas seulement la decision binaire.

    Brier Score = (1/n) * somme des (probabilite_predite - valeur_reelle)^2

Un service meteo ne dit pas "il pleuvra demain" - il dit "70% de risque de pluie". Le Brier Score mesure si ces probabilites sont bien calibrees. Si le modele annonce 70% pour 100 jours, il devrait effectivement pleuvoir environ 70 fois parmi ces 100 jours.

- Brier Score = 0 : predictions parfaites
- Brier Score = 0.25 : equivalent a un modele aleatoire sur une classe equilibree
- Plus la valeur est basse, mieux c'est

#### ROC-AUC

Mesure la capacite discriminante du modele pour tous les seuils possibles. AUC = 1 est parfait, AUC = 0.5 est aleatoire. C'est une metrique independante du seuil de decision.

#### Optimisation du seuil de decision

Par defaut, un modele predit "Rain" si proba > 0.5. Ce seuil n'est pas toujours optimal pour maximiser le F2-score. On cherche le seuil qui maximise le F2 sur la courbe Precision-Rappel.

In [ ]:
y_pred  = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]

# Seuil optimal pour F2
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
f2_by_thresh = (1 + 2**2) * precisions[:-1] * recalls[:-1] / (2**2 * precisions[:-1] + recalls[:-1] + 1e-9)
best_idx    = np.argmax(f2_by_thresh)
best_thresh = thresholds[best_idx]
y_pred_opt  = (y_proba >= best_thresh).astype(int)

f2_default = fbeta_score(y_test, y_pred, beta=2)
f2_opt     = fbeta_score(y_test, y_pred_opt, beta=2)
f1_opt     = f1_score(y_test, y_pred_opt)
roc        = roc_auc_score(y_test, y_proba)
brier      = brier_score_loss(y_test, y_proba)
logloss    = log_loss(y_test, y_proba)

report = classification_report(y_test, y_pred_opt, output_dict=True)

print('=' * 58)
print(f'RESULTATS FINAUX - {best_name} | Split chronologique')
print('=' * 58)
print(f'Seuil optimise         : {best_thresh:.3f} (au lieu de 0.5)')
print(f'F2-score (seuil opt.)  : {f2_opt:.4f}   <- metrique principale')
print(f'F2-score (seuil 0.5)   : {f2_default:.4f}')
print(f'F1-score (seuil opt.)  : {f1_opt:.4f}')
print(f'Recall  Rain           : {report["1"]["recall"]:.4f}')
print(f'Precision Rain         : {report["1"]["precision"]:.4f}')
print(f'ROC-AUC                : {roc:.4f}')
print(f'Brier Score Loss       : {brier:.4f}  (plus bas = mieux)')
print(f'Log Loss               : {logloss:.4f}')
print('=' * 58)
print()
print(classification_report(y_test, y_pred_opt, target_names=['No Rain','Rain']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Courbe Precision-Rappel
axes[0].plot(recalls[:-1], precisions[:-1], color='#4C72B0', lw=2, label='Courbe P-R')
axes[0].scatter(recalls[best_idx], precisions[best_idx], color='red', s=120, zorder=5,
                label=f'Seuil opt={best_thresh:.2f}\nF2={f2_opt:.4f}')
axes[0].fill_between(recalls[:-1], precisions[:-1], alpha=0.1, color='#4C72B0')
axes[0].set_xlabel('Recall (Sensibilite)')
axes[0].set_ylabel('Precision')
axes[0].set_title('Courbe Precision-Rappel')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred_opt)
ConfusionMatrixDisplay(cm, display_labels=['No Rain','Rain']).plot(ax=axes[1], colorbar=False, cmap='Blues')
fn = ((y_test==1) & (y_pred_opt==0)).sum()
fp = ((y_test==0) & (y_pred_opt==1)).sum()
axes[1].set_title(f'Matrice de confusion\nFaux Negatifs : {fn} | Faux Positifs : {fp}')

# Courbe de calibration (Brier)
frac_pos, mean_pred = calibration_curve(y_test, y_proba, n_bins=10)
axes[2].plot(mean_pred, frac_pos, marker='o', color='#4C72B0', lw=2, label=f'{best_name}\nBrier={brier:.4f}')
axes[2].plot([0,1],[0,1],'k--', lw=1.5, label='Calibration parfaite')
axes[2].set_xlabel('Probabilite predite')
axes[2].set_ylabel('Frequence reelle de pluie')
axes[2].set_title('Courbe de calibration\n(la diagonale = probabilites parfaites)')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle(f'Evaluation finale - {best_name} | F2={f2_opt:.4f} | ROC-AUC={roc:.4f}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 12.2 - Analyse des erreurs

In [ ]:
results_df = X_test.copy()
results_df['y_true']     = y_test.values
results_df['y_pred']     = y_pred_opt
results_df['y_proba']    = y_proba
results_df['error_type'] = 'Correct'
results_df.loc[(results_df['y_true']==1)&(results_df['y_pred']==0), 'error_type'] = 'Faux Negatif'
results_df.loc[(results_df['y_true']==0)&(results_df['y_pred']==1), 'error_type'] = 'Faux Positif'

fn_df = results_df[results_df['error_type'] == 'Faux Negatif']
fp_df = results_df[results_df['error_type'] == 'Faux Positif']

print(f'Faux Negatifs (pluie manquee) : {len(fn_df):,} ({len(fn_df)/len(y_test)*100:.1f}%)')
print(f'Faux Positifs (fausse alarme) : {len(fp_df):,} ({len(fp_df)/len(y_test)*100:.1f}%)')
print(f'Predictions correctes         : {(results_df["error_type"]=="Correct").sum():,} ({(results_df["error_type"]=="Correct").sum()/len(y_test)*100:.1f}%)')

print('\nProfil des Faux Negatifs (jours de pluie non detectes) :')
cols_show = ['Humidity3pm', 'Pressure9am', 'Rainfall', 'y_proba']
cols_show = [c for c in cols_show if c in fn_df.columns]
print(fn_df[cols_show].describe().round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for label, color, name in [(0,'#4C72B0','No Rain reel'), (1,'#DD8452','Rain reel')]:
    subset = results_df[results_df['y_true']==label]['y_proba']
    axes[0].hist(subset, bins=40, alpha=0.55, color=color, label=name, density=True)
axes[0].axvline(best_thresh, color='red', lw=2, linestyle='--',
                label=f'Seuil={best_thresh:.2f}')
axes[0].set_xlabel('Probabilite predite Rain')
axes[0].set_ylabel('Densite')
axes[0].set_title('Distribution des probabilites par classe reelle')
axes[0].legend(fontsize=9)

axes[1].hist(fn_df['y_proba'], bins=30, alpha=0.7, color='#C44E52',
             label=f'Faux Negatifs ({len(fn_df)})', density=True)
axes[1].hist(fp_df['y_proba'], bins=30, alpha=0.7, color='#4C72B0',
             label=f'Faux Positifs ({len(fp_df)})', density=True)
axes[1].set_xlabel('Probabilite predite')
axes[1].set_title('Probabilites des erreurs FN vs FP')
axes[1].legend()

plt.suptitle('Analyse des erreurs', fontsize=11)
plt.tight_layout()
plt.show()
print('Les Faux Negatifs ont des probabilites proches du seuil.')
print('Ce sont des cas meteorologiquement ambigus, difficiles a ameliorer sans donnees supplementaires.')

### 12.3 - Importance des features

In [ ]:
model_step        = final_model.named_steps['model']
selector_step     = final_model.named_steps['selector']
preprocessor_step = final_model.named_steps['preprocessor']

ohe_cols_final = preprocessor_step.named_transformers_['cat']['ohe'].get_feature_names_out(categorical_features)
all_feat_names = np.array(numeric_features + list(ohe_cols_final))
selected_mask  = selector_step.get_support()
selected_names = all_feat_names[selected_mask]

if hasattr(model_step, 'feature_importances_'):
    importances = model_step.feature_importances_
    feat_imp = pd.DataFrame({'feature': selected_names, 'importance': importances})
    feat_imp = feat_imp.sort_values('importance', ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(10, 7))
    colors_imp = ['#DD8452' if i < 5 else '#4C72B0' for i in range(len(feat_imp))]
    ax.barh(feat_imp['feature'][::-1], feat_imp['importance'][::-1],
            color=colors_imp[::-1], alpha=0.85)
    ax.set_title(f'Top 20 Feature Importances - {best_name}', fontsize=11)
    ax.set_xlabel('Importance relative')
    plt.tight_layout()
    plt.show()

    print('Top 10 features les plus importantes :')
    for _, row in feat_imp.head(10).iterrows():
        print(f'  {row["feature"]:<35} {row["importance"]:.4f}')

---
## Synthese finale

### Decisions cles et justifications

| Etape | Decision | Justification |
|-------|----------|---------------|
| EDA | Analyser avant de modeliser | Comprendre les donnees evite les erreurs coteuses |
| Cleaning | Aberrations physiques uniquement | Trop nettoyer = perte de signal utile |
| Feature Engineering | Sin/cos cyclique + derivees temporelles | Encode la physique meteorologique |
| Split | Chronologique (pas aleatoire) | Donnees temporelles : pas de regard vers le futur |
| Preprocessing | RobustScaler + Pipeline | Resistant aux outliers, pas de data leakage |
| Ponderation | sample_weight heuristique | Reduit l'impact du bruit de label |
| Feature Selection | Mutual Information | Capture les relations non-lineaires |
| Resampling | Aucun (pas de SMOTE) | Donnees temporelles, interpolations non physiques |
| Metrique | F2-score (pas F1) | Un Faux Negatif coute plus cher qu'un Faux Positif |
| Calibration | Brier Score + courbe | Les probabilites comptent autant que la decision binaire |

In [ ]:
print('=' * 62)
print('SYNTHESE FINALE - WEATHER AUSTRALIA')
print('=' * 62)
print(f'Modele selectionne     : {best_name}')
print(f'Split                  : Chronologique')
print(f'Seuil optimise         : {best_thresh:.3f}')
print()
print('METRIQUES SUR TEST SET (jamais vu pendant l\'entrainement) :')
print(f'  F2-score             : {f2_opt:.4f}   <- metrique principale')
print(f'  F1-score             : {f1_opt:.4f}')
print(f'  Recall Rain          : {report["1"]["recall"]:.4f}')
print(f'  Precision Rain       : {report["1"]["precision"]:.4f}')
print(f'  ROC-AUC              : {roc:.4f}')
print(f'  Brier Score Loss     : {brier:.4f}')
print(f'  Faux Negatifs        : {fn:,} ({fn/len(y_test)*100:.1f}%)')
print(f'  Faux Positifs        : {fp:,} ({fp/len(y_test)*100:.1f}%)')
print('=' * 62)
print()
print('Pistes d\'amelioration :')
print('- Ajouter des donnees satellite ou radar (Sunshine tres incompleta)')
print('- Essayer CatBoost (gere nativement les categories sans OHE)')
print('- Entrainer un modele par zone climatique')
print('- Calibration des probabilites (CalibratedClassifierCV)')